# Gold: Sales by Product

**Objetivo:** consolidar métricas de vendas por produto, país e dia, em inglês — permitindo ao CFO (ou qualquer consumidor executivo) analisar performance de vendas por SKU (ex: "quanto vendemos de Mayonnaise no Brasil, dia a dia").

**Fonte:** tabela Delta `silver.fato_vendas`, enriquecida com `silver.produtos` (nome e tamanho) e tradução de país.

**Destino:** tabela Delta `gold.sales_by_product`.

**Granularidade:** produto + país + dia (`period`).

**Observação:** mesma limitação de câmbio já documentada — dias sem cotação disponível resultam em `total_usd` nulo.

In [0]:
# Imports e configuração do schema

from pyspark.sql.functions import col, sum as spark_sum, round as spark_round, when

spark.sql("CREATE SCHEMA IF NOT EXISTS poc_latam_food.gold")

print("Schema 'gold' verificado/criado com sucesso.")

In [0]:
# JOIN com Produtos + tradução do país

df_fato_vendas = spark.table("poc_latam_food.silver.fato_vendas")
df_produtos_silver = spark.table("poc_latam_food.silver.produtos")

df_vendas_enriquecido = (
    df_fato_vendas.alias("v")
    .join(
        df_produtos_silver.alias("p"),
        col("v.produto_id") == col("p.produto_id"),
        "left"
    )
    .withColumn(
        "country",
        when(col("v.pais") == "BRASIL", "Brazil")
        .when(col("v.pais") == "ARGENTINA", "Argentina")
        .when(col("v.pais") == "MEXICO", "Mexico")
        .otherwise(col("v.pais"))
    )
    .select(
        col("p.nome_ingles").alias("product_name"),
        col("p.tamanho").alias("size"),
        "country",
        col("v.data_venda").alias("period"),
        col("v.quantidade").alias("quantity"),
        col("v.valor_total_usd").alias("total_usd")
    )
)

print("JOIN configurado.")

In [0]:
# Agregação por produto, tamanho, país e dia

df_gold_by_product = (
    df_vendas_enriquecido
    .groupBy("product_name", "size", "country", "period")
    .agg(
        spark_sum("quantity").alias("quantity_sold"),
        spark_round(spark_sum("total_usd"), 2).cast("decimal(15,2)").alias("total_usd")
    )
    .orderBy("period", "country", "product_name", "size")
)

df_gold_by_product.display()

In [0]:
# Gravação - overwrite completo (recálculo)

(
    df_gold_by_product
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("poc_latam_food.gold.sales_by_product")
)

print("Gold sales_by_product concluído.")

In [0]:
# Validação visual - gold.sales_by_product

spark.table("poc_latam_food.gold.sales_by_product").display()